# KAN Semi-Infinite-Domain Hyperparameter Optimization

Optuna searches KAN hyperparameters for the semi-infinite manufactured problem.

In [7]:
import pandas as pd

In [8]:
import os
import sys
from datetime import datetime
from importlib import reload

current_dir = os.getcwd()
utilities_dir = os.path.join(current_dir, '../../utils')
os.chdir(current_dir)
if utilities_dir not in sys.path:
    sys.path.insert(0, utilities_dir)

import joblib
import optuna
import pandas as pd
import torch
import pinns
import pinns_semi_infinite
import semi_infinite
from pinns_semi_infinite import run_experiment_semi_inf

reload(pinns)
reload(semi_infinite)
reload(pinns_semi_infinite)
torch.set_default_dtype(torch.float32)
set_seed(1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

## Optuna Search Configuration

In [9]:
KAN_SEARCH_SPACE = {
    'hidden_layers': [1, 2, 3],
    'hidden_units': [15, 25, 35],
    'grid_size': [3, 5, 7],
    'spline_order': [2, 3, 4],
    'learning_rate': [1e-4, 1e-3, 1e-2],
}

N_TRIALS = 50
ADAM_ITERS = 2000
LBFGS_ITERS = 2000
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
results_dir = f'results_kan_semi_infinite_optuna_{timestamp}'
os.makedirs(results_dir, exist_ok=True)
print(f'Results will be saved to: {results_dir}')
print(f'Optuna trials: {N_TRIALS}')

Results will be saved to: results_kan_semi_infinite_optuna_2026-09-15_21-51-22
Optuna trials: 50


## Objective Function

In [10]:
def objective(trial):
    config = {
        name: trial.suggest_categorical(name, values)
        for name, values in KAN_SEARCH_SPACE.items()
    }
        
    print(
        f'\n--- Trial {trial.number}: '
        f"L={config['hidden_layers']}, N={config['hidden_units']}, "
        f"grid={config['grid_size']}, order={config['spline_order']}, "
        f"lr={config['learning_rate']:.0e} ---"
    )

    try:
        result = run_experiment_semi_inf(
            model_type='KAN',
            hidden_layers=config['hidden_layers'],
            hidden_units=config['hidden_units'],
            grid_size=config['grid_size'],
            spline_order=config['spline_order'],
            adam_lr=config['learning_rate'],
            device=device,
            adam_iters=ADAM_ITERS,
            lbfgs_iters=LBFGS_ITERS,
            results_dir=results_dir,
        )
    except Exception as error:
        print(f'Trial {trial.number} failed: {error}')
        raise optuna.exceptions.TrialPruned() from error

    err_u = float(result['err_u_global'])
    err_k = float(result['err_k_global'])
    compute_time = float(result['compute_time_sec'])
    mean_global_error = 0.5 * (err_u + err_k)
    trial.set_user_attr('err_u', err_u)
    trial.set_user_attr('err_k', err_k)
    trial.set_user_attr('compute_time_sec', compute_time)
    print(
        f'Success! Time: {compute_time:.2f}s | '
        f'Err U: {err_u:.3e} | Err K: {err_k:.3e} | '
        f'Mean error: {mean_global_error:.3e}'
    )
    return mean_global_error

## Run Optimization

In [ ]:
sampler = optuna.samplers.TPESampler(seed=1)
study = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    study_name=f'kan_semi_infinite_domain_{timestamp}',
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    catch=(RuntimeError, ValueError),
)

print('\n========================================')
print('BEST KAN SEMI-INFINITE CONFIGURATION')
print('========================================')
print(f'Mean global error: {study.best_value:.6e}')
print('Parameters:')
for name, value in study.best_params.items():
    print(f'  {name}: {value}')

[I 2026-09-15 21:51:25,329] A new study created in memory with name: kan_semi_infinite_domain_2026-09-15_21-51-22



--- Trial 0: L=2, N=15, grid=7, order=4, lr=1e-03 ---


In [ ]:
data_dir = os.path.join(results_dir, 'data')
os.makedirs(data_dir, exist_ok=True)
joblib.dump(study, os.path.join(data_dir, 'study.pkl'))
joblib.dump(study, os.path.join(data_dir, f'study_{timestamp}.pkl'))
study_df = study.trials_dataframe()
study_csv_path = os.path.join(data_dir, 'study.csv')
study_df.to_csv(study_csv_path, index=False)
filtered_df = study_df[
    (study_df['params_hidden_layers'] == 3)
    & (study_df['params_hidden_units'] == 25)
].sort_values(by='value', ascending=True)
filtered_csv_path = os.path.join(data_dir, 'study_filtered_sorted.csv')
filtered_df.to_csv(filtered_csv_path, index=False)
print(f'Saved study to: {data_dir}')
print(f'Saved trial summary to: {study_csv_path}')
print(f'Saved filtered summary to: {filtered_csv_path}')

Saved study to: results_kan_semi_infinite_optuna_2026-09-15_21-41-21/data
Saved trial summary to: results_kan_semi_infinite_optuna_2026-09-15_21-41-21/data/study.csv
Saved filtered summary to: results_kan_semi_infinite_optuna_2026-09-15_21-41-21/data/study_filtered_sorted.csv
